# 用 Claude Managed Agents 构建数据分析智能体

## 引言

每个团队都有那样一个人：被人递过来一个 CSV，然后被问"这里面有什么有意思的东西？"。在本 Cookbook 中，你会构建一个替他回答问题的智能体：上传一个 CSV，得到一份带交互式图表的叙述性 HTML 报告。

你会在 [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview) 上运行它，这是 Anthropic 为有状态、使用工具的智能体提供的托管运行时，建立在四个核心概念之上：

- **智能体（Agent）**：模型、系统提示词、工具、MCP 服务器和技能
- **环境（Environment）**：一个配置好的容器模板（依赖包、网络访问）
- **会话（Session）**：环境中一个正在运行的智能体实例，执行某个具体任务并生成输出
- **事件（Events）**：你的应用与智能体之间交换的消息（用户轮次、工具结果、状态更新）

一个智能体加一个环境就得到一个会话。你把数据作为资源挂到它上面，然后通过发送事件并读回流来驱动它。

Anthropic 替你处理沙箱、工具执行和上下文管理。如果你需要对智能体循环和部署的完全控制，请改用 [Claude Agent SDK](https://platform.claude.com/docs/en/api/agent-sdk/overview)。

### 你将学到什么

读完本 Cookbook 你将能够：

- 为数据分析搭建一个可复用的环境和智能体
- 把数据集交给智能体处理
- 在智能体运行时观察它的进展
- 下载报告以及它产生的其他文件

### 前置条件

- Python 3.11+
- 来自 [控制台](https://platform.claude.com/settings/keys) 的 Anthropic API 密钥，设置为 `ANTHROPIC_API_KEY`

安装依赖：

In [10]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [11]:
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv, set_key

load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-4-6"

## 1. 创建环境

**环境**是一份可复用的容器规格。在这里声明 `pandas` 和 `plotly`，意味着每个会话开始时它们都已预装，所以智能体可以立即开始分析，而不必先运行 `pip install`。

这里的网络是 `unrestricted`，以便智能体从 plotly 的 CDN 加载它——但这让它能访问互联网上的任何地方，所以在生产环境中请改用[主机白名单](https://platform.claude.com/docs/en/managed-agents/environments)。

In [12]:
env = client.beta.environments.create(
    name="cookbook-data-analyst-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        "packages": {
            "type": "packages",
            "pip": ["pandas", "plotly"],
        },
    },
)

## 2. 创建智能体

**智能体**把一个模型与一个系统提示词和一组工具配对。输出质量大部分来自系统提示词；这里的提示词强调叙述结构、由具体数字支撑的结论，以及在一个 HTML 文件中嵌入多张 plotly 图表的正确做法。

[`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools) 提供八个工具：`bash`、`read`、`write`、`edit`、`glob`、`grep`、`web_fetch` 和 `web_search`。这里它们都在 `always_allow` 下运行，两个网络工具被禁用，因为本次分析是离线的。

In [13]:
ANALYST_SYSTEM_PROMPT = """\
You are a senior data analyst producing a publication-quality report.

## Style
- Professional and precise. Let the data speak with concrete numbers.
- Short paragraphs (2-3 sentences) between charts.
- Lead with the most actionable finding.

## Execution
- Write .py scripts and run them with `python3 script.py`.
- Sample large tables (`nrows=` / `.sample()`) instead of loading everything.
- Sanity-check key metrics before building narrative around them.

## Charts
- Build each chart as its own `go.Figure()`, embed with
  `fig.to_html(include_plotlyjs=False, full_html=False)`, and load plotly
  from the CDN once in <head>.
- Always set `marker_color` and `template='simple_white'`.

## Output
Write a single self-contained `report.html` to /mnt/session/outputs/
with inline CSS, 3+ embedded plotly charts, and a closing section of
actionable recommendations. Confirm "Saved: report.html" when done.
"""

agent = client.beta.agents.create(
    name="cookbook-data-analyst",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/data-analyst-agent"},
    model=MODEL,
    system=ANALYST_SYSTEM_PROMPT,
    tools=[
        {
            "type": "agent_toolset_20260401",
            # default_config applies to every tool in the set;
            # entries in configs override specific tools.
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
)

## 3. 上传数据集

随附的示例 CSV 有 50 行，因此分析在几分钟内完成。这里可以换成任意 CSV（或一个 CSV 的 zip 压缩包），后续流程完全相同。

In [14]:
DATA_PATH = Path("example_data/data_analyst_agent/sales_data.csv")

with DATA_PATH.open("rb") as f:
    dataset = client.beta.files.upload(file=(DATA_PATH.name, f, "text/csv"))

print(f"Uploaded {DATA_PATH.name} ({dataset.size_bytes} bytes) as {dataset.id}")

Uploaded sales_data.csv (2833 bytes) as file_011CZqKKomgL45BLo2J8K5X9


## 4. 创建会话并发送任务

**会话**把智能体绑定到环境以及任何挂载的文件。传入 `{"type": "agent", "id": ..., "version": ...}` 会复用上面创建的带版本智能体。`resources` 把上传的文件挂载到容器内给定的绝对路径。

创建会话后，发送一个带任务的 `user.message` 事件。智能体会立即开始工作。

In [15]:
MOUNT_PATH = f"/mnt/session/uploads/{DATA_PATH.name}"

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": dataset.id, "mount_path": MOUNT_PATH}],
    title="Sales analysis",
)

ANALYSIS_PROMPT = f"""\
Analyze the e-commerce orders in {MOUNT_PATH}.

Columns: order_id, customer_id, product, category, price, quantity,
order_date, region.

Focus on revenue by category and region, repeat-customer behavior, and
one surprising pattern. Produce report.html per your system instructions.
"""

client.beta.sessions.events.send(
    session.id,
    events=[
        {"type": "user.message", "content": [{"type": "text", "text": ANALYSIS_PROMPT}]},
    ],
)
print(f"Session {session.id} running")

Session sesn_011CZqKKqyPRph9J5khKLn6u running


## 5. 流式查看运行过程

在[控制台](https://platform.claude.com/)的 **Sessions** 下打开会话，实时查看每个事件、工具使用和词元数量：

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/data_analyst_agent/console_session.png" alt="Session trace in the Console" width="700" />

下面的辅助函数跟踪同一份事件流，在 `agent.message` 文本和 `agent.tool_use` 调用到达时打印它们，并在 `session.status_idle` 时返回。

In [ ]:
def wait_for_idle(session_id: str) -> None:
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            for block in ev.content:
                if block.type == "text":
                    text = block.text
                    print(text[:300] + ("..." if len(text) > 300 else ""))
        elif t in ("agent.tool_use", "agent.mcp_tool_use"):
            print(f"  [{ev.name}]")
        elif t == "session.status_idle":
            return
        elif t == "session.status_terminated":
            raise RuntimeError(
                "Session terminated before going idle. "
                f"Trace: https://platform.claude.com/sessions/{session_id}"
            )


wait_for_idle(session.id)

## 6. 取回报告

智能体写入 `/mnt/session/outputs/` 的任何内容都会被持久化，并通过 Files API 以 `scope_id=<session_id>` 呈现。写入容器其他位置的文件不会被持久化。

[Files API](https://platform.claude.com/docs/en/api/beta/files/list) 是 beta 中的一项独立功能，所以要在这里使用 `scope_id`，你还需要传入 Managed Agents 的 beta 请求头。

In [17]:
outputs = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in outputs.data:
    print(f.filename, f.size_bytes)

# The list also includes the mounted input CSV; pick out the report.
report = next((f for f in outputs.data if f.filename == "report.html"), None)
if report is None:
    raise RuntimeError(f"report.html not found. Files: {[f.filename for f in outputs.data]}")
content = client.beta.files.download(report.id)
Path("report.html").write_bytes(content.read())
print("Downloaded report.html")

report.html 53728
sales_data.csv 2833
Downloaded report.html


## 7. 清理与下一步

智能体和环境只创建一次，在多次运行中复用；每次对话都创建一个新的会话。既然报告已经拿到，归档这个会话以释放它的容器。下面的代码行把智能体 ID 和环境 ID 保存到 `.env`，以便 [`slack_data_bot.ipynb`](slack_data_bot.ipynb) 用它们启动新会话。

> **警告：** 在运行下一个单元格之前，确认 `.env` 已列入 `.gitignore`——绝不要提交它。

In [18]:
client.beta.sessions.archive(session.id)

set_key(".env", "ANALYST_ENV_ID", env.id)
set_key(".env", "ANALYST_AGENT_ID", agent.id)
set_key(".env", "ANALYST_AGENT_VERSION", str(agent.version))
print("Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env")

Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env


你端到端地构建并运行了一个数据分析智能体：可复用的环境和智能体、挂载了你 CSV 的会话、实时的事件流，以及下载下来的 HTML 报告。

从这里出发：

- 打开下载的 `report.html`，查看叙述内容和图表。
- 在[控制台](https://platform.claude.com/)打开会话，检查词元用量和完整的事件日志。
- 继续前往 [`slack_data_bot.ipynb`](slack_data_bot.ipynb)，从 Slack 驱动这个智能体。